In [38]:
# 1. Importar librerías

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
from sklearn.utils import resample




In [39]:
# 2. Cargar dataset limpio

DATA_PATH = (r"C:\Users\aljoz\Desktop\Proyectos Desafio Profesional\02. Diabetes\diabetic_data_clean.csv")

df = pd.read_csv(DATA_PATH)

print("Dimensiones:", df.shape)



Dimensiones: (101766, 54)


In [40]:
# 3. Variable objetivo: readmisión <30 días

df["readm_30"] = (df["readmitted"] == "<30").astype(int)

print("Tasa base readmisión:", round(df["readm_30"].mean(), 4))


Tasa base readmisión: 0.1116


In [41]:
# 4. Ingeniería de variables

# Edad numérica
age_bounds = df["age"].str.extract(r"\[(\d+)-(\d+)\)")
df["age_num"] = (age_bounds[0].astype(int) + age_bounds[1].astype(int)) / 2

# Variables clínicas
df["insulin_bin"] = (df["insulin"] != "No").astype(int)
df["change_bin"] = (df["change"] == "Ch").astype(int)

# Historial asistencial
df["high_inpatient"] = (df["number_inpatient"] >= 2).astype(int)
df["high_emergency"] = (df["number_emergency"] >= 1).astype(int)

df["utilization_score"] = (
    df["number_inpatient"] +
    df["number_emergency"] +
    df["number_outpatient"]
)

In [42]:
# 5. Definir conjunto final de variables

features = [
    "age_num",
    "number_diagnoses",
    "time_in_hospital",
    "num_medications",
    "insulin_bin",
    "change_bin",
    "high_inpatient",
    "high_emergency",
    "utilization_score"
]

X = df[features]
y = df["readm_30"]


In [43]:
# 6. Modelo Random Forest optimizado

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

probs = rf.predict_proba(X_test)[:, 1]

# AUC principal
auc = roc_auc_score(y_test, probs)


# IC95% del AUC por bootstrap

n_bootstraps = 1000
rng = np.random.RandomState(42)
bootstrapped_scores = []

for _ in range(n_bootstraps):
    indices = rng.randint(0, len(probs), len(probs))
    
    if len(np.unique(y_test.iloc[indices])) < 2:
        continue
        
    score = roc_auc_score(y_test.iloc[indices], probs[indices])
    bootstrapped_scores.append(score)

sorted_scores = np.sort(bootstrapped_scores)
ic_inf = sorted_scores[int(0.025 * len(sorted_scores))]
ic_sup = sorted_scores[int(0.975 * len(sorted_scores))]

print("AUC Modelo Final:", round(auc, 4))
print("IC95%:", f"({round(ic_inf,4)} – {round(ic_sup,4)})")

AUC Modelo Final: 0.6396
IC95%: (0.629 – 0.6504)


In [44]:

# 8. Comparación de importancias

# 8.1. Importancia por permutación en modelo completo
perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=8,
    random_state=42,
    n_jobs=-1
)

importance_full = (
    pd.Series(perm.importances_mean, index=features)
      .sort_values(ascending=False)
)

print("Importancia variables - Modelo completo")
print(importance_full.round(4))


# 8.2. Peso relativo de high_inpatient
peso_relativo = (
    abs(importance_full["high_inpatient"]) /
    abs(importance_full).sum()
) * 100

print("Peso relativo high_inpatient (%):", round(peso_relativo, 2))

Importancia variables - Modelo completo
high_inpatient       0.0331
utilization_score    0.0057
age_num             -0.0012
insulin_bin         -0.0018
change_bin          -0.0048
high_emergency      -0.0049
time_in_hospital    -0.0085
number_diagnoses    -0.0097
num_medications     -0.0129
dtype: float64
Peso relativo high_inpatient (%): 40.14


In [45]:
# 9. Modelo sin ingresos hospitalarios previos

df_no_prev = df[df["number_inpatient"] == 0].copy()

X_np = df_no_prev[features]
y_np = df_no_prev["readm_30"]

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np,
    y_np,
    test_size=0.25,
    stratify=y_np,
    random_state=42
)

rf_np = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_np.fit(X_train_np, y_train_np)

probs_np = rf_np.predict_proba(X_test_np)[:, 1]

auc_np = roc_auc_score(y_test_np, probs_np)


# IC95% del AUC por bootstrap

n_bootstraps = 1000
rng = np.random.RandomState(42)
boot_scores = []

for _ in range(n_bootstraps):
    indices = rng.randint(0, len(probs_np), len(probs_np))
    
    if len(np.unique(y_test_np.iloc[indices])) < 2:
        continue
    
    score = roc_auc_score(y_test_np.iloc[indices], probs_np[indices])
    boot_scores.append(score)

boot_scores = np.sort(np.array(boot_scores))

ic_inf = np.percentile(boot_scores, 2.5)
ic_sup = np.percentile(boot_scores, 97.5)

print("AUC sin ingresos previos:", round(auc_np, 4))
print("IC95%:", f"({round(ic_inf,4)} – {round(ic_sup,4)})")


AUC sin ingresos previos: 0.5802
IC95%: (0.5658 – 0.5959)
